In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "utils").is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError("Project root with src/utils was not found")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from utils import (
    DEFAULT_CUBE_SIZES,
    BereaPatchDataset,
    DigitalCorePipeline,
    FiLMRoutedUNet3D,
    check_required_dependencies,
)

check_required_dependencies()
device = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = ROOT / "outputs" / "networks"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "device:", device)


ROOT: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct device: cuda


In [2]:
CUBE_SIZES = list(DEFAULT_CUBE_SIZES)
SPLIT = "train"
MAX_SAMPLES = 20
USE_TARGET_MASK = True
THRESHOLD = 0.5
VOXEL_SIZE_M = 2.25e-6
SEG_CHECKPOINT = ROOT / "models" / "film_routed_unet3d_best.pth"


In [3]:
dataset = BereaPatchDataset(ROOT, split=SPLIT, cube_size=CUBE_SIZES, use_raw_gray=False, noise_types=["none"], balance=True)
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

seg_model = None
if not USE_TARGET_MASK:
    seg_model = FiLMRoutedUNet3D(base_channels=16, ctx_dim=64).to(device)
    checkpoint = torch.load(SEG_CHECKPOINT, map_location=device)
    seg_model.load_state_dict(checkpoint["model_state_dict"])

pipeline = DigitalCorePipeline(
    segmentation_model=seg_model,
    device=device,
    threshold=THRESHOLD,
    voxel_size=VOXEL_SIZE_M,
    mu=1.0e-3,
)
print("cube sizes:", CUBE_SIZES)
print("samples:", len(dataset))
display(dataset.df.groupby(["rock", "cube_size", "split"]).size().rename("samples").reset_index())


cube sizes: [64, 128, 192]
samples: 19662


,rock,cube_size,split,samples
0,BanderaBrown,64,train,3277
1,BanderaBrown,128,train,410
2,BanderaBrown,192,train,173
3,Berea,64,train,3277
4,Berea,128,train,410
5,Berea,192,train,173


In [5]:
rows = []

for sample_id, batch in enumerate(tqdm(loader, desc="extract networks")):
    if MAX_SAMPLES is not None and sample_id >= MAX_SAMPLES:
        break

    if USE_TARGET_MASK:
        cube = batch["y"][0, 0].numpy() > 0.5
        input_is_pore_mask = True
    else:
        cube = batch["x"][0].numpy()
        input_is_pore_mask = False

    cube_size = int(batch["cube_size"][0])
    domain_size = (cube_size * VOXEL_SIZE_M, cube_size * VOXEL_SIZE_M, cube_size * VOXEL_SIZE_M)
    rock = batch["rock"][0]

    try:
        result = pipeline.run_cube(
            cube,
            input_is_pore_mask=input_is_pore_mask,
            domain_size=domain_size,
            include_ph=True,
            compute_openpnm_baseline=True,
        )
    except Exception as exc:
        print(f"sample {sample_id} skipped: {exc}")
        continue

    coord = batch["coord"][0].tolist()
    result.network.metadata.update({
        "sample_id": sample_id,
        "coord": coord,
        "rock": rock,
        "cube_size": cube_size,
        "openpnm_k": result.permeability.k_openpnm,
    })
    out_path = OUT_DIR / f"network_{SPLIT}_{sample_id:04d}.pt"
    torch.save(result.network, out_path)

    row = {
        "sample_id": sample_id,
        "path": str(out_path),
        "rock": rock,
        "cube_size": cube_size,
        "z": coord[0],
        "y": coord[1],
        "x": coord[2],
        **result.network.metadata,
    }
    if result.permeability.k_openpnm is not None:
        row.update({f"openpnm_{k}": v for k, v in result.permeability.k_openpnm.items()})
    rows.append(row)

summary = pd.DataFrame(rows)
summary_path = OUT_DIR / f"network_{SPLIT}_summary.csv"
summary.to_csv(summary_path, index=False)
summary


extract networks:   0%|          | 20/19662 [06:38<108:46:09, 19.94s/it]


,sample_id,path,rock,cube_size,z,y,x,num_pores,num_throats,node_feature_dim,edge_feature_dim,ph_summary,topology,coord,openpnm_k,openpnm_kx,openpnm_ky,openpnm_kz
0,0,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,Berea,64,768,256,768,69,78,21,16,"[68.0, 0.001469602226279676, 4.678990444517694...","{'percolates_z': True, 'percolates_y': True, '...","[768, 256, 768]","{'kx': 6.028989302257268e-15, 'ky': 2.59439111...",6.028989e-15,2.594391e-14,2.790282e-14
1,1,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,Berea,128,0,768,512,480,763,21,16,"[479.0, 0.01131247915327549, 5.744593363488093...","{'percolates_z': True, 'percolates_y': True, '...","[0, 768, 512]","{'kx': 1.6699107607001163e-13, 'ky': 1.0578056...",1.669911e-13,1.057806e-13,1.671371e-13
2,2,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,BanderaBrown,64,576,256,936,102,127,21,16,"[101.0, 0.002056662691757083, 4.10787142754998...","{'percolates_z': True, 'percolates_y': True, '...","[576, 256, 936]","{'kx': 2.7184128993419334e-13, 'ky': 1.9876009...",2.718413e-13,1.987601e-13,2.745000e-14
3,3,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,Berea,192,0,192,768,1073,1673,21,16,"[511.0, 0.017328308895230293, 9.56197618506848...","{'percolates_z': True, 'percolates_y': True, '...","[0, 192, 768]","{'kx': 1.2613528506263075e-13, 'ky': 8.0431196...",1.261353e-13,8.043120e-14,1.104811e-13
4,4,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,Berea,64,896,64,768,94,119,21,16,"[93.0, 0.001879242598079145, 5.044177669333294...","{'percolates_z': True, 'percolates_y': True, '...","[896, 64, 768]","{'kx': 2.371871899986448e-13, 'ky': 5.35170996...",2.371872e-13,5.351710e-13,1.841662e-13
5,5,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,BanderaBrown,128,640,0,872,613,836,21,16,"[511.0, 0.011882705613970757, 5.43003916391171...","{'percolates_z': True, 'percolates_y': True, '...","[640, 0, 872]","{'kx': 4.1082414572814093e-14, 'ky': 1.8486049...",4.108241e-14,1.848605e-14,2.737502e-14
6,6,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,Berea,64,768,384,640,84,111,21,16,"[83.0, 0.0017708807718008757, 4.63464311906136...","{'percolates_z': True, 'percolates_y': True, '...","[768, 384, 640]","{'kx': 3.9734664669891635e-13, 'ky': 4.2744873...",3.973466e-13,4.274487e-13,3.648574e-13
7,7,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,BanderaBrown,192,576,0,768,1690,2601,21,16,"[511.0, 0.01894703134894371, 7.999330409802496...","{'percolates_z': True, 'percolates_y': True, '...","[576, 0, 768]","{'kx': 5.417963885328436e-14, 'ky': 5.41432949...",5.417964e-14,5.414329e-14,6.082614e-14
8,8,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,BanderaBrown,192,808,384,0,1472,2134,21,16,"[511.0, 0.018872013315558434, 7.17153889127075...","{'percolates_z': True, 'percolates_y': True, '...","[808, 384, 0]","{'kx': 2.87055282560997e-14, 'ky': 3.776419528...",2.870553e-14,3.776420e-14,2.734856e-14
9,9,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,BanderaBrown,192,192,808,384,1564,2471,21,16,"[511.0, 0.018460212275385857, 8.49397983984090...","{'percolates_z': True, 'percolates_y': True, '...","[192, 808, 384]","{'kx': 8.886346404708082e-14, 'ky': 7.62543537...",8.886346e-14,7.625435e-14,7.295066e-14
